# XAI Playground Research Analysis

This notebook rebuilds research summaries from the local `./logs` directory. It is intentionally broad: it extracts experiment metadata, generation outputs, latest TCF evaluation results, TCF judge rows, question-normalization rows, summary tables, and PDF plots under `./plots`.

The notebook assumes it is run from the repository root.


## Setup


In [ ]:

from __future__ import annotations

import json
import math
import os
from datetime import datetime
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
LOGS_DIR = ROOT / "logs"
PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

TCF_NAMES = {"TCF", "trace_coverage_faithfulness", "faithfulness_score"}
REPRESENTATIVE_CASES = {
    (0, 0), (5, 5), (10, 10), (2, 7), (0, 10),
    (10, 0), (2, 2), (8, 5), (5, 8), (9, 9),
}

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.bbox": "tight",
    "font.size": 10,
})

print(f"Repository root: {ROOT}")
print(f"Logs directory: {LOGS_DIR}")
print(f"Plots directory: {PLOTS_DIR}")


## Load and Normalize Logs


In [ ]:

def read_json(path: Path) -> Any:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def to_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        if isinstance(value, str) and value.lower() in {"nan", "none", ""}:
            return None
        result = float(str(value).replace(",", "."))
    except Exception:
        return None
    return None if math.isnan(result) else result


def to_int(value: Any) -> int | None:
    numeric = to_float(value)
    return int(numeric) if numeric is not None else None


def parse_timestamp(value: Any) -> Any:
    return pd.to_datetime(str(value), errors="coerce", utc=True) if value else pd.NaT


def first_non_empty(*values: Any) -> Any:
    for value in values:
        if value not in (None, "", []):
            return value
    return None


def trace_model_from_path(trace_path: Any) -> str | None:
    if not trace_path:
        return None
    parts = Path(str(trace_path)).parts
    return parts[-2] if len(parts) >= 2 else None


def parse_trace_scores_from_path(trace_path: Any) -> tuple[int | None, int | None]:
    if not trace_path:
        return (None, None)
    import re
    match = re.search(r"food(?P<food>-?\d+)_service(?P<service>-?\d+)", Path(str(trace_path)).stem)
    return (int(match.group("food")), int(match.group("service"))) if match else (None, None)


def evaluation_history(data: dict[str, Any]) -> list[dict[str, Any]]:
    evaluations = data.get("evaluations")
    if isinstance(evaluations, list):
        clean = [item for item in evaluations if isinstance(item, dict)]
        if clean:
            return sorted(clean, key=lambda item: str(item.get("timestamp_utc") or ""), reverse=True)
    legacy = data.get("evaluation")
    return [legacy] if isinstance(legacy, dict) and legacy else []


def is_tcf_evaluation(evaluation: dict[str, Any]) -> bool:
    return str(evaluation.get("metric_short_name") or "") in TCF_NAMES or str(evaluation.get("metric_name") or "") in TCF_NAMES


def tcf_history(data: dict[str, Any]) -> list[dict[str, Any]]:
    return [evaluation for evaluation in evaluation_history(data) if is_tcf_evaluation(evaluation)]


def latest_tcf(data: dict[str, Any]) -> dict[str, Any] | None:
    history = tcf_history(data)
    return history[0] if history else None


def status_bucket_from_tcf(evaluation: dict[str, Any] | None) -> str:
    if not evaluation:
        return "missing"
    status = str(evaluation.get("status") or "unknown")
    if status == "success" and to_float(evaluation.get("score")) is not None:
        return "success"
    return status if status in {"error", "skipped"} else "missing"


def quality_bucket(score: float | None) -> str:
    if score is None:
        return "unscored"
    if score >= 0.8:
        return "good >=0.80"
    if score >= 0.4:
        return "mixed 0.40-0.79"
    return "bad <0.40"


def row_from_log(path: Path, data: dict[str, Any]) -> dict[str, Any]:
    generation_parameters = data.get("generation_parameters") or {}
    backend = data.get("backend") or {}
    metrics = data.get("metrics") or {}
    trace = data.get("trace") or {}
    trace_meta = trace.get("path_metadata") or {}
    trace_summary = trace.get("summary") or {}
    inputs = trace_summary.get("inputs") or {}
    outputs = trace_summary.get("outputs") or {}
    tip = outputs.get("tip") if isinstance(outputs.get("tip"), dict) else {}
    trace_path = trace.get("path")
    path_food, path_service = parse_trace_scores_from_path(trace_path)
    output = data.get("output") or {}
    output_text = str(output.get("text") or "")
    response_metadata = output.get("response_metadata") or {}
    usage = metrics.get("usage") or response_metadata.get("usage") or {}
    tcf = latest_tcf(data)
    tcf_hist = tcf_history(data)
    tcf_score = to_float((tcf or {}).get("score")) if tcf else None
    rulebase = data.get("rulebase") or {}
    cli_params = (data.get("cli") or {}).get("parameters") or {}
    timestamp = parse_timestamp(data.get("timestamp_utc"))
    food = to_int(first_non_empty(trace_meta.get("food_score"), inputs.get("food"), path_food))
    service = to_int(first_non_empty(trace_meta.get("service_score"), inputs.get("service"), path_service))
    return {
        "path": str(path),
        "filename": path.name,
        "date_dir": path.parent.name,
        "timestamp": timestamp,
        "date": timestamp.date().isoformat() if pd.notna(timestamp) else None,
        "framework_version": data.get("framework_version"),
        "status": str(data.get("status") or "unknown"),
        "mode": str(data.get("mode") or "unknown"),
        "mode_description": data.get("mode_description"),
        "backend": str(backend.get("name") or cli_params.get("backend") or "unknown"),
        "model": str(generation_parameters.get("model") or "unknown"),
        "temperature": to_float(generation_parameters.get("temperature")),
        "top_p": to_float(generation_parameters.get("top_p")),
        "max_tokens": to_int(generation_parameters.get("max_tokens")),
        "timeout_seconds": to_int(generation_parameters.get("timeout_seconds")),
        "trace_model": first_non_empty(trace_meta.get("model_name"), trace_model_from_path(trace_path), "unknown"),
        "trace_path": trace_path,
        "food": food,
        "service": service,
        "tip": to_float((tip or {}).get("value")),
        "trace_format": trace.get("format"),
        "prompt_char_count": to_int(metrics.get("prompt_char_count")),
        "prompt_message_count": to_int(metrics.get("prompt_message_count")),
        "example_count": to_int(metrics.get("example_count")),
        "active_proposition_count": to_int(metrics.get("active_proposition_count")),
        "rulebase_rule_count": to_int(metrics.get("rulebase_rule_count") or rulebase.get("rule_count")),
        "rulebase_source": rulebase.get("source"),
        "rulebase_method": rulebase.get("method"),
        "generation_time_seconds": to_float(metrics.get("generation_time_seconds")),
        "output_char_count": to_int(metrics.get("output_char_count") or len(output_text)),
        "output_word_count": to_int(metrics.get("output_word_count") or len(output_text.split())),
        "output_text_present": bool(output_text.strip()),
        "prompt_tokens": to_int(usage.get("prompt_tokens")),
        "completion_tokens": to_int(usage.get("completion_tokens")),
        "total_tokens": to_int(usage.get("total_tokens")),
        "finish_reason": response_metadata.get("finish_reason"),
        "error_type": ((data.get("error") or {}).get("type")),
        "error_message": ((data.get("error") or {}).get("message")),
        "tcf_status": status_bucket_from_tcf(tcf),
        "tcf_score": tcf_score,
        "tcf_quality_bucket": quality_bucket(tcf_score),
        "tcf_yes_count": to_int((tcf or {}).get("yes_count")),
        "tcf_total_questions": to_int((tcf or {}).get("total_questions")),
        "tcf_trace_fact_count": to_int((tcf or {}).get("trace_fact_count")),
        "tcf_eval_count": len(tcf_hist),
        "tcf_latest_timestamp": parse_timestamp((tcf or {}).get("timestamp_utc")) if tcf else pd.NaT,
        "tcf_judge_model": str(((tcf or {}).get("generation_parameters") or {}).get("model") or ""),
        "tcf_judge_backend": str(((tcf or {}).get("backend") or {}).get("name") or ""),
        "representative_case": (food, service) in REPRESENTATIVE_CASES,
    }


def judge_rows_from_log(path: Path, data: dict[str, Any]) -> list[dict[str, Any]]:
    base = row_from_log(path, data)
    tcf = latest_tcf(data)
    if not tcf:
        return []
    rows = []
    for result in ((tcf.get("judge") or {}).get("results") or []):
        if not isinstance(result, dict):
            continue
        rows.append({
            "path": str(path), "filename": path.name, "model": base["model"], "mode": base["mode"], "trace_model": base["trace_model"],
            "food": base["food"], "service": base["service"], "section": result.get("section"), "line_id": result.get("line_id"),
            "answer": result.get("answer"), "is_yes": bool(result.get("is_yes")), "elapsed_seconds": to_float(result.get("elapsed_seconds")),
            "question": result.get("question"), "trace_line": result.get("trace_line"), "reason": result.get("reason"),
        })
    return rows


def normalization_rows_from_log(path: Path, data: dict[str, Any]) -> list[dict[str, Any]]:
    base = row_from_log(path, data)
    tcf = latest_tcf(data)
    if not tcf:
        return []
    rows = []
    for item in ((tcf.get("question_generation") or {}).get("question_normalization") or []):
        if not isinstance(item, dict):
            continue
        rows.append({
            "path": str(path), "filename": path.name, "model": base["model"], "mode": base["mode"], "trace_model": base["trace_model"],
            "line_id": item.get("line_id"), "section": item.get("section"), "replaced": bool(item.get("replaced")),
            "replacement_reason": item.get("replacement_reason"), "raw_question": item.get("raw_question"), "normalized_question": item.get("normalized_question"),
        })
    return rows



EXPECTED_EXPERIMENT_COLUMNS = [
    "path", "filename", "date_dir", "timestamp", "date", "framework_version", "status", "mode",
    "mode_description", "backend", "model", "temperature", "top_p", "max_tokens", "timeout_seconds",
    "trace_model", "trace_path", "food", "service", "tip", "trace_format", "prompt_char_count",
    "prompt_message_count", "example_count", "active_proposition_count", "rulebase_rule_count",
    "rulebase_source", "rulebase_method", "generation_time_seconds", "output_char_count",
    "output_word_count", "output_text_present", "prompt_tokens", "completion_tokens", "total_tokens",
    "finish_reason", "error_type", "error_message", "tcf_status", "tcf_score", "tcf_quality_bucket",
    "tcf_yes_count", "tcf_total_questions", "tcf_trace_fact_count", "tcf_eval_count",
    "tcf_latest_timestamp", "tcf_judge_model", "tcf_judge_backend", "representative_case",
]

def coerce_loaded_frames(
    experiments: pd.DataFrame,
    judges: pd.DataFrame,
    normalization: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if experiments.empty:
        experiments = pd.DataFrame(columns=EXPECTED_EXPERIMENT_COLUMNS)
    for column in EXPECTED_EXPERIMENT_COLUMNS:
        if column not in experiments.columns:
            experiments[column] = np.nan
    experiments["timestamp"] = pd.to_datetime(experiments["timestamp"], utc=True, errors="coerce")
    experiments["tcf_latest_timestamp"] = pd.to_datetime(experiments["tcf_latest_timestamp"], utc=True, errors="coerce")
    for column in ["food", "service", "tip", "generation_time_seconds", "output_word_count", "output_char_count", "prompt_tokens", "completion_tokens", "total_tokens", "tcf_score", "tcf_yes_count", "tcf_total_questions", "tcf_eval_count"]:
        if column in experiments.columns:
            experiments[column] = pd.to_numeric(experiments[column], errors="coerce")
    if "representative_case" in experiments.columns:
        experiments["representative_case"] = experiments["representative_case"].fillna(False).astype(bool)
    if "output_text_present" in experiments.columns:
        experiments["output_text_present"] = experiments["output_text_present"].fillna(False).astype(bool)
    return experiments, judges, normalization


def load_sidecar_frames() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    experiment_csv = PLOTS_DIR / "experiment_summary_rows.csv"
    judge_csv = PLOTS_DIR / "tcf_judge_rows.csv"
    normalization_csv = PLOTS_DIR / "tcf_question_normalization_rows.csv"
    if not experiment_csv.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    experiments = pd.read_csv(experiment_csv)
    judges = pd.read_csv(judge_csv) if judge_csv.exists() else pd.DataFrame()
    normalization = pd.read_csv(normalization_csv) if normalization_csv.exists() else pd.DataFrame()
    print(f"Loaded portable sidecar data from {PLOTS_DIR}")
    return coerce_loaded_frames(experiments, judges, normalization)


def load_all() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    log_paths = sorted(LOGS_DIR.glob("*/*.json"))
    if not log_paths:
        return load_sidecar_frames()

    experiment_rows = []
    judge_rows = []
    normalization_rows = []
    for path in log_paths:
        data = read_json(path)
        if not isinstance(data, dict):
            continue
        experiment_rows.append(row_from_log(path, data))
        judge_rows.extend(judge_rows_from_log(path, data))
        normalization_rows.extend(normalization_rows_from_log(path, data))
    experiments = pd.DataFrame(experiment_rows)
    judges = pd.DataFrame(judge_rows)
    normalization = pd.DataFrame(normalization_rows)
    print(f"Loaded raw experiment logs from {LOGS_DIR}")
    return coerce_loaded_frames(experiments, judges, normalization)

df, judges, normalization = load_all()
success = df[df["status"] == "success"].copy()
scored = df[(df["tcf_status"] == "success") & df["tcf_score"].notna()].copy()
errors = df[df["status"] != "success"].copy()
print(f"Loaded {len(df)} experiment rows, {len(success)} successful runs, {len(scored)} scored latest TCF runs.")


## Summary Tables


In [ ]:

def agg_summary(group_col: str) -> pd.DataFrame:
    rows = []
    for name, group in df.groupby(group_col, dropna=False):
        succ = group[group["status"] == "success"]
        group_scored = group[group["tcf_score"].notna()]
        rows.append({
            group_col: name,
            "logs": len(group),
            "success": int((group["status"] == "success").sum()),
            "success_rate": (group["status"] == "success").mean(),
            "scored": len(group_scored),
            "mean_tcf": group_scored["tcf_score"].mean() if len(group_scored) else np.nan,
            "median_tcf": group_scored["tcf_score"].median() if len(group_scored) else np.nan,
            "mean_time_s": succ["generation_time_seconds"].mean() if len(succ) else np.nan,
            "mean_words": succ["output_word_count"].mean() if len(succ) else np.nan,
            "mean_total_tokens": succ["total_tokens"].mean() if len(succ) else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["logs", group_col], ascending=[False, True])

headline = pd.DataFrame([
    {"metric": "total_logs", "value": len(df)},
    {"metric": "successful_logs", "value": len(success)},
    {"metric": "non_success_logs", "value": len(errors)},
    {"metric": "scored_latest_tcf_logs", "value": len(scored)},
    {"metric": "mean_latest_tcf", "value": scored["tcf_score"].mean() if len(scored) else np.nan},
    {"metric": "median_latest_tcf", "value": scored["tcf_score"].median() if len(scored) else np.nan},
    {"metric": "perfect_latest_tcf_logs", "value": int((scored["tcf_score"].round(12) == 1.0).sum()) if len(scored) else 0},
    {"metric": "nonperfect_latest_tcf_logs", "value": int((scored["tcf_score"].round(12) != 1.0).sum()) if len(scored) else 0},
    {"metric": "unique_models", "value": df["model"].nunique()},
    {"metric": "unique_modes", "value": df["mode"].nunique()},
    {"metric": "unique_rulebases", "value": df["trace_model"].nunique()},
    {"metric": "unique_food_service_cases", "value": len(set(zip(df["food"], df["service"])))},
])

model_summary = agg_summary("model")
mode_summary = agg_summary("mode")
rulebase_summary = agg_summary("trace_model")
backend_summary = agg_summary("backend")
mode_model_summary = df.groupby(["model", "mode"], dropna=False).agg(
    logs=("path", "count"),
    success=("status", lambda s: int((s == "success").sum())),
    scored=("tcf_score", lambda s: int(s.notna().sum())),
    mean_tcf=("tcf_score", "mean"),
    mean_words=("output_word_count", "mean"),
).reset_index().sort_values(["model", "mode"])

status_summary = df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="logs")
tcf_status_summary = df["tcf_status"].value_counts(dropna=False).rename_axis("tcf_status").reset_index(name="logs")
quality_summary = df["tcf_quality_bucket"].value_counts(dropna=False).rename_axis("quality_bucket").reset_index(name="logs")
error_summary = errors.groupby(["error_type", "error_message"], dropna=False).size().reset_index(name="logs").sort_values("logs", ascending=False)
nonperfect = scored[scored["tcf_score"].round(12) != 1.0].sort_values(["tcf_score", "path"])

for name, table in [
    ("headline", headline), ("status", status_summary), ("tcf_status", tcf_status_summary), ("quality", quality_summary),
    ("model", model_summary), ("mode", mode_summary), ("rulebase", rulebase_summary), ("backend", backend_summary),
    ("mode_model", mode_model_summary), ("errors", error_summary.head(20)), ("nonperfect_tcf", nonperfect.head(100)),
]:
    print("\n", name)
    display(table)


## TCF Judge and Question-Normalization Analysis


In [ ]:

if not judges.empty:
    no_judgments = judges[judges["is_yes"] == False].copy()
    no_by_section = no_judgments.groupby(["section", "model", "mode"], dropna=False).size().reset_index(name="no_judgments").sort_values("no_judgments", ascending=False)
    judge_latency = judges.groupby(["section"], dropna=False)["elapsed_seconds"].agg(["count", "mean", "median", "max"]).reset_index()
    display(no_by_section.head(50))
    display(judge_latency)
else:
    print("No judge rows found.")

if not normalization.empty:
    normalization_summary = normalization.groupby(["replaced", "replacement_reason"], dropna=False).size().reset_index(name="questions").sort_values("questions", ascending=False)
    display(normalization_summary)
else:
    print("No question-normalization rows found. Older logs may predate this TCF refinement.")


## Generate PDF Plots


In [ ]:

def savefig(name: str) -> Path:
    path = PLOTS_DIR / name
    plt.savefig(path)
    plt.close()
    return path

plot_files = []

if not df.empty and df["date"].notna().any():
    ax = df.pivot_table(index="date", columns="status", values="path", aggfunc="count", fill_value=0).sort_index().plot(kind="bar", stacked=True, figsize=(10, 5), width=0.85)
    ax.set_title("Experiment logs by date and status"); ax.set_xlabel("Date"); ax.set_ylabel("Logs")
    plot_files.append(savefig("experiment_status_by_date.pdf"))

if not df.empty:
    counts = df["model"].value_counts().sort_values(ascending=True)
    ax = counts.plot(kind="barh", figsize=(9, max(4, 0.35 * len(counts))))
    ax.set_title("Experiment logs by generation model"); ax.set_xlabel("Logs"); ax.set_ylabel("Model")
    plot_files.append(savefig("experiments_by_model.pdf"))

    success_rate = df.assign(success=df["status"].eq("success")).groupby("model")["success"].mean().sort_values()
    ax = success_rate.plot(kind="barh", figsize=(9, max(4, 0.35 * len(success_rate))))
    ax.set_title("Success rate by generation model"); ax.set_xlabel("Success rate"); ax.set_ylabel("Model"); ax.set_xlim(0, 1)
    plot_files.append(savefig("success_rate_by_model.pdf"))

if not success.empty:
    pivot = success.pivot_table(index="model", columns="mode", values="path", aggfunc="count", fill_value=0)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 5))
    ax.set_title("Successful explanations by model and mode"); ax.set_xlabel("Model"); ax.set_ylabel("Successful logs"); ax.tick_params(axis="x", rotation=30)
    plot_files.append(savefig("successful_explanations_by_model_mode.pdf"))

    pivot = success.pivot_table(index="trace_model", columns="mode", values="path", aggfunc="count", fill_value=0)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(9, 5))
    ax.set_title("Successful explanations by rulebase and mode"); ax.set_xlabel("Rulebase / trace model"); ax.set_ylabel("Successful logs"); ax.tick_params(axis="x", rotation=0)
    plot_files.append(savefig("successful_explanations_by_rulebase_mode.pdf"))

if not scored.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(scored["tcf_score"], bins=np.linspace(0, 1, 21), edgecolor="white")
    ax.set_title("TCF score distribution"); ax.set_xlabel("Latest TCF score"); ax.set_ylabel("Logs")
    plot_files.append(savefig("tcf_score_distribution.pdf"))

    for group_col, filename, title in [("model", "tcf_by_model_boxplot.pdf", "TCF by generation model"), ("mode", "tcf_by_mode_boxplot.pdf", "TCF by experiment mode"), ("trace_model", "tcf_by_rulebase_boxplot.pdf", "TCF by rulebase / trace model")]:
        groups = [(name, group["tcf_score"].dropna().to_numpy()) for name, group in scored.groupby(group_col)]
        groups = [(name, values) for name, values in groups if len(values)]
        if groups:
            fig, ax = plt.subplots(figsize=(max(8, 0.8 * len(groups)), 5))
            ax.boxplot([values for _, values in groups], tick_labels=[name for name, _ in groups], showmeans=True)
            ax.set_title(title); ax.set_ylabel("Latest TCF score"); ax.set_ylim(-0.02, 1.02); ax.tick_params(axis="x", rotation=30)
            plot_files.append(savefig(filename))

    pivot = scored.pivot_table(index="model", columns="mode", values="tcf_score", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(pivot))))
    image = ax.imshow(pivot.fillna(np.nan), vmin=0, vmax=1, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right"); ax.set_yticks(range(len(pivot.index)), pivot.index); ax.set_title("Mean TCF by model and mode")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]
            if pd.notna(value): ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white" if value < 0.65 else "black")
    fig.colorbar(image, ax=ax, label="Mean TCF")
    plot_files.append(savefig("mean_tcf_heatmap_model_mode.pdf"))

    pivot = scored.pivot_table(index="food", columns="service", values="tcf_score", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(pivot.sort_index(ascending=False), vmin=0, vmax=1, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)), [int(x) if float(x).is_integer() else x for x in pivot.columns]); ax.set_yticks(range(len(pivot.index)), [int(x) if float(x).is_integer() else x for x in pivot.sort_index(ascending=False).index])
    ax.set_xlabel("Service score"); ax.set_ylabel("Food score"); ax.set_title("Mean TCF by food/service case"); fig.colorbar(image, ax=ax, label="Mean TCF")
    plot_files.append(savefig("mean_tcf_heatmap_food_service.pdf"))

    fig, ax = plt.subplots(figsize=(8, 5))
    for mode, group in scored.groupby("mode"):
        ax.scatter(group["output_word_count"], group["tcf_score"], alpha=0.65, label=mode, s=28)
    ax.set_title("TCF vs explanation length"); ax.set_xlabel("Output words"); ax.set_ylabel("Latest TCF score"); ax.set_ylim(-0.02, 1.02); ax.legend(fontsize=8)
    plot_files.append(savefig("tcf_vs_output_words.pdf"))

    if scored["tcf_judge_model"].replace("", np.nan).notna().any():
        pivot = scored.groupby("tcf_judge_model")["tcf_score"].agg(["count", "mean"]).sort_values("mean")
        ax = pivot["mean"].plot(kind="barh", figsize=(9, max(4, 0.35 * len(pivot))))
        ax.set_title("Mean TCF by judge model"); ax.set_xlabel("Mean latest TCF"); ax.set_ylabel("Judge model"); ax.set_xlim(0, 1)
        plot_files.append(savefig("mean_tcf_by_judge_model.pdf"))

if not success.empty and success["tip"].notna().any():
    pivot = success.pivot_table(index="food", columns="service", values="tip", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(pivot.sort_index(ascending=False), aspect="auto", cmap="magma")
    ax.set_xticks(range(len(pivot.columns)), [int(x) if float(x).is_integer() else x for x in pivot.columns]); ax.set_yticks(range(len(pivot.index)), [int(x) if float(x).is_integer() else x for x in pivot.sort_index(ascending=False).index])
    ax.set_xlabel("Service score"); ax.set_ylabel("Food score"); ax.set_title("Mean fuzzy tip output by logged food/service case"); fig.colorbar(image, ax=ax, label="Tip (%)")
    plot_files.append(savefig("mean_tip_heatmap_food_service.pdf"))

if not success.empty and success["generation_time_seconds"].notna().any():
    summary = success.groupby("model")["generation_time_seconds"].mean().sort_values()
    ax = summary.plot(kind="barh", figsize=(9, max(4, 0.35 * len(summary))))
    ax.set_title("Mean generation time by model"); ax.set_xlabel("Seconds"); ax.set_ylabel("Model")
    plot_files.append(savefig("mean_generation_time_by_model.pdf"))

if not success.empty and success["output_word_count"].notna().any():
    summary = success.groupby("model")["output_word_count"].mean().sort_values()
    ax = summary.plot(kind="barh", figsize=(9, max(4, 0.35 * len(summary))))
    ax.set_title("Mean explanation length by model"); ax.set_xlabel("Words"); ax.set_ylabel("Model")
    plot_files.append(savefig("mean_output_words_by_model.pdf"))

if not success.empty and success["total_tokens"].notna().any():
    summary = success.groupby("model")[["prompt_tokens", "completion_tokens"]].mean().dropna(how="all")
    summary = summary.loc[summary.sum(axis=1).sort_values(ascending=False).index]
    ax = summary.plot(kind="bar", stacked=True, figsize=(11, 5))
    ax.set_title("Mean token usage by model"); ax.set_xlabel("Model"); ax.set_ylabel("Tokens"); ax.tick_params(axis="x", rotation=30)
    plot_files.append(savefig("mean_token_usage_by_model.pdf"))

if not errors.empty:
    counts = errors["error_type"].fillna("unknown").value_counts().sort_values(ascending=True)
    ax = counts.plot(kind="barh", figsize=(8, 4))
    ax.set_title("Non-success logs by error type"); ax.set_xlabel("Logs"); ax.set_ylabel("Error type")
    plot_files.append(savefig("error_types.pdf"))

if not judges.empty:
    no_rows = judges[judges["is_yes"] == False]
    if not no_rows.empty:
        counts = no_rows["section"].fillna("unknown").value_counts().sort_values(ascending=True)
        ax = counts.plot(kind="barh", figsize=(7, 4))
        ax.set_title("TCF NO judgments by trace section"); ax.set_xlabel("NO judgments"); ax.set_ylabel("Section")
        plot_files.append(savefig("tcf_no_judgments_by_section.pdf"))

if not df.empty:
    summary = df.groupby("model")["tcf_eval_count"].mean().sort_values()
    ax = summary.plot(kind="barh", figsize=(9, max(4, 0.35 * len(summary))))
    ax.set_title("Mean TCF evaluation history count by model"); ax.set_xlabel("Evaluations per log"); ax.set_ylabel("Model")
    plot_files.append(savefig("mean_tcf_evaluation_count_by_model.pdf"))

if not success.empty:
    coverage = success.groupby("trace_model").apply(lambda g: len(set(zip(g["food"], g["service"]))))
    coverage = coverage.sort_values()
    ax = coverage.plot(kind="barh", figsize=(8, 4))
    ax.set_title("Unique food/service cases covered by rulebase"); ax.set_xlabel("Unique cases"); ax.set_ylabel("Rulebase / trace model")
    plot_files.append(savefig("case_coverage_by_rulebase.pdf"))

if not normalization.empty:
    counts = normalization.assign(replaced=normalization["replaced"].map(bool)).groupby("replaced")["path"].count()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["accepted", "replaced"], [int(counts.get(False, 0)), int(counts.get(True, 0))])
    ax.set_title("TCF generated-question normalization"); ax.set_ylabel("Questions")
    plot_files.append(savefig("tcf_question_normalization.pdf"))

pd.DataFrame({"plot": [str(path) for path in plot_files]}).to_csv(PLOTS_DIR / "generated_plot_index.csv", index=False)
df.to_csv(PLOTS_DIR / "experiment_summary_rows.csv", index=False)
if not judges.empty: judges.to_csv(PLOTS_DIR / "tcf_judge_rows.csv", index=False)
if not normalization.empty: normalization.to_csv(PLOTS_DIR / "tcf_question_normalization_rows.csv", index=False)

print(f"Wrote {len(plot_files)} PDF plots to {PLOTS_DIR}")
display(pd.DataFrame({"plot": [path.name for path in plot_files]}))
